In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

**DimUser**

### **Autoloader**

In [0]:

df_user = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimUser/checkpoint")\
    .option("schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@spotifyazureketaki.dfs.core.windows.net/DimUser")

print(df_user.schema)


In [0]:
display(df_user,checkpointLocation="abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimUser/checkpoint")

In [0]:
df_bronze = spark.read \
    .format("parquet") \
    .load("abfss://bronze@spotifyazureketaki.dfs.core.windows.net/DimUser")
display(df_bronze)

In [0]:
df_update = df_bronze.withColumn("user_name",upper(col("user_name")))
display(df_update)

In [0]:
/Workspace/Users/kolad.river.rafting2025@gmail.com/SpotifyAzureProject/spotify_dab/utils/transformations.py

In [0]:
import os 
import sys 
project_pth = os.path.join(os.getcwd(),'..','..')
sys.path.append(project_pth)

In [0]:
project_pth

In [0]:
from utils.transformations import reusable

In [0]:
df_user_obj = reusable()
df_user = df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])
display(df_user.schema)

In [0]:
print(spark.conf.get("spark.sql.shuffle.partitions"))

In [0]:
print("Shuffle partitions:",
      spark.conf.get("spark.sql.shuffle.partitions"))

print("Is streaming DataFrame:",
      df_user.isStreaming)

In [0]:
df_user.writeStream.format("delta")

In [0]:

df_user.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimUser/checkpoint_v2"
    ) \
    .trigger(once=True) \
    .start(
        "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimUser/data"
    )


### **DimArtist**

In [0]:

df_art = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimArtist/ckpt_DA")\
    .option("schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@spotifyazureketaki.dfs.core.windows.net/DimArtist")

print(df_art.schema)

In [0]:
display(df_art,checkpointLocation="abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimArtist/ckpt_DA")

In [0]:
df_art_obj = reusable()
df_art = df_art_obj.dropColumns(df_art,['_rescued_data'])
df_art = df_art.dropDuplicates(['artist_id'])
display(df_art.schema)

In [0]:
df_art.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimArtist/checkpoint"
    ) \
    .trigger(once=True) \
    .start(
        "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimArtist/data"
    )

### **DimTrack**

In [0]:
df_track = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimTrack/ckpt_DA")\
    .option("schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@spotifyazureketaki.dfs.core.windows.net/DimTrack")
print(df_track.schema)

In [0]:
display(df_track,checkpointLocation="abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimTrack/ckpt_DA")

In [0]:
from pyspark.sql.functions import col, when, translate

# Create duration categories
df_track = df_track.withColumn(
    "durationFlag",
    when(col("duration_sec") < 150, "low")
    .when(col("duration_sec") < 300, "medium")
    .otherwise("high")
)

# Replace hyphens with spaces in track_name
df_track = df_track.withColumn(
    "track_name",
    translate(col("track_name"), "-", " ")
)

# Display the result
display(df_track, checkpointLocation='abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimTrack/ckpt_DT')

In [0]:
print(df_track.isStreaming)

In [0]:
display(df_track.schema)

In [0]:
df_track = reusable().dropColumns(df_track,['_rescued_data'])
display(df_track.schema)

In [0]:
df_track.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimTrack/checkpoint"
    ) \
    .trigger(once=True) \
    .start(
        "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimTrack/data"
    )

### **DimDate**

In [0]:
df_date = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimDate/ckpt_DA")\
    .option("schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@spotifyazureketaki.dfs.core.windows.net/DimDate")
print(df_date.schema)

In [0]:
display(df_track,checkpointLocation="abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimDate/ckpt_DA")

In [0]:
df_date = reusable().dropColumns(df_date,['_rescued_data'])
display(df_date.schema)

In [0]:
df_date.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimDate/checkpoint"
    ) \
    .trigger(once=True) \
    .start(
        "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimDate/data"
    )

### **FactStream**

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@spotifyazureketaki.dfs.core.windows.net/FactStream/ckpt_FS")\
    .option("schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@spotifyazureketaki.dfs.core.windows.net/FactStream")
print(df_fact.schema)

In [0]:
display(df_fact,checkpointLocation="abfss://silver@spotifyazureketaki.dfs.core.windows.net/FactStream/ckpt_FS")

In [0]:
display(df_fact.schema)

In [0]:
df_fact = reusable().dropColumns(df_fact,['_rescued_data'])
display(df_fact.schema)

In [0]:
df_fact.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@spotifyazureketaki.dfs.core.windows.net/FactStream/checkpoint"
    ) \
    .trigger(once=True) \
    .start(
        "abfss://silver@spotifyazureketaki.dfs.core.windows.net/FactStream/data"
    )

In [0]:
display(dbutils.fs.ls(
    "abfss://silver@spotifyazureketaki.dfs.core.windows.net/"
))

In [0]:
display(dbutils.fs.ls(
    "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimArtist/"
))

In [0]:
display(dbutils.fs.ls(
    "abfss://silver@spotifyazureketaki.dfs.core.windows.net/FactStream/"
))

In [0]:
display(dbutils.fs.ls(
    "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimArtist/"
))

In [0]:
display(dbutils.fs.ls(
    "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimArtist/data/"
))

In [0]:
artist_path = "abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimArtist/data/"
df_artist = spark.read.format("delta").load(artist_path)
print("Artist row count:", df_artist.count())
display(df_artist)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS spotify_cata.silver.DimArtist
USING DELTA
LOCATION 'abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimArtist/data/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS spotify_cata.silver.DimDate
USING DELTA
LOCATION 'abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimDate/data/';

CREATE TABLE IF NOT EXISTS spotify_cata.silver.DimTrack
USING DELTA
LOCATION 'abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimTrack/data/';

CREATE TABLE IF NOT EXISTS spotify_cata.silver.DimUser
USING DELTA
LOCATION 'abfss://silver@spotifyazureketaki.dfs.core.windows.net/DimUser/data/';

CREATE TABLE IF NOT EXISTS spotify_cata.silver.FactStream
USING DELTA
LOCATION 'abfss://silver@spotifyazureketaki.dfs.core.windows.net/FactStream/data/';